# Experiment 15 — Flat-type segmented AutoGluon residual models

Layer 1 is the saved Exp 14 AutoGluon rent ensemble. OOF residuals are deterministically divided into four observable flat-type segments: executive, 5-room, 4-room, and 1–3-room. Each segment receives its own AutoGluon GBM + CatBoost residual ensemble. Segment-specific residual weights are selected on the latest OOF fold, then the experts are refit on all OOF residual rows and evaluated on untouched outer validation.

In [1]:
from pathlib import Path
import json,os,sys
import numpy as np
import pandas as pd
from autogluon.tabular import TabularPredictor
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
from sklearn.model_selection import TimeSeriesSplit
configured_root=os.environ.get('KDDM_PROJECT_ROOT')
roots=([Path(configured_root)] if configured_root else [])+[Path.cwd(),*Path.cwd().parents]
project_root=next((root for root in roots if (root/'exploration/metul/preprocessing.py').exists()),None)
if project_root is None:raise FileNotFoundError('Set KDDM_PROJECT_ROOT to the project directory.')
sys.path.insert(0,str(project_root/'exploration'/'metul'))
from preprocessing import RentalPreprocessor,load_datasets
TARGET='MONTHLY_RENT';RESIDUAL_TARGET='layer1_residual';N_SPLITS=3
TIME_LIMIT=int(os.environ.get('EXP15_TIME_LIMIT','300'))
RUN_ID=os.environ.get('EXP15_RUN_ID',pd.Timestamp.now().strftime('%Y%m%d_%H%M%S'))
HYPERPARAMETERS={'GBM':{},'CAT':{}}
source_nb=json.loads((project_root/'exploration'/'metul'/'exp_14_resale_features_lgbm.ipynb').read_text())
source_cells={cell.get('metadata',{}).get('id'):''.join(cell.get('source',[])) for cell in source_nb['cells']}
exec(source_cells['setup'],globals());exec(source_cells['exp9-best-feature-transformer'],globals())
model_root=project_root/'exploration'/'metul'/'ag_exp_14_ag_residual_models'
requested_run=os.environ.get('EXP15_SOURCE_RUN_ID')
if requested_run:source_run=model_root/requested_run
else:
    candidates=sorted(path for path in model_root.iterdir() if (path/'layer1_full').exists() and (path/'layer1_oof_3').exists())
    if not candidates:raise FileNotFoundError('No completed Exp 14 run found.')
    source_run=candidates[-1]
output_root=project_root/'exploration'/'metul'/'ag_exp_15_flat_type_residuals'/RUN_ID
print(f'Loading Layer 1 from {source_run.name}; saving residual experts to {output_root}')


Loading Layer 1 from 20261015_144824; saving residual experts to /Users/metul/Desktop/KDDM-project/exploration/metul/ag_exp_15_flat_type_residuals/20261016_182615


In [2]:
_,datasets=load_datasets(project_root/'data')
raw=datasets['train'].copy();raw['period']=pd.to_datetime(raw['RENT_APPROVAL_DATE']+'-01').dt.to_period('M')
periods=np.sort(raw.period.unique());outer_cutoff=periods[int(len(periods)*.80)]
raw_fit,raw_valid=raw.loc[raw.period<outer_cutoff].copy(),raw.loc[raw.period>=outer_cutoff].copy()
def best_features(history,future):
    processor=RentalPreprocessor(datasets)
    history_base=processor.fit_transform(history.drop(columns=['period',TARGET],errors='ignore'))
    future_base=processor.transform(future.drop(columns=['period',TARGET],errors='ignore'))
    return make_exp9_best_features(history,future,history_base,future_base)
def predictions_and_disagreement(predictor,frame):
    prediction=predictor.predict(frame).astype(float).to_numpy()
    names=[name for name in predictor.model_names() if ('LightGBM' in name or 'CatBoost' in name) and 'WeightedEnsemble' not in name]
    components=np.vstack([predictor.predict(frame,model=name).astype(float).to_numpy() for name in names])
    disagreement=components.max(axis=0)-components.min(axis=0) if len(components)>1 else np.zeros(len(frame))
    return prediction,disagreement
def residual_segment(flat_type):
    value=str(flat_type).strip().lower()
    if value=='executive':return 'executive'
    if value=='5 room':return '5_room'
    if value=='4 room':return '4_room'
    return '1_to_3_room'
fit_periods=np.sort(raw_fit.period.unique());parts=[]
for fold,(early_index,later_index) in enumerate(TimeSeriesSplit(n_splits=N_SPLITS).split(fit_periods),1):
    early=raw_fit.loc[raw_fit.period.isin(fit_periods[early_index])].copy();later=raw_fit.loc[raw_fit.period.isin(fit_periods[later_index])].copy()
    _,later_x=best_features(early,later);model=TabularPredictor.load(str(source_run/f'layer1_oof_{fold}'))
    prediction,disagreement=predictions_and_disagreement(model,later_x)
    part=later_x.copy();part['layer1_prediction']=prediction;part['layer1_disagreement']=disagreement;part[RESIDUAL_TARGET]=later[TARGET].to_numpy()-prediction;part['fold']=fold;part['segment']=part.FLAT_TYPE.map(residual_segment)
    parts.append(part);print(f'Fold {fold}: {len(part):,} residual rows')
oof=pd.concat(parts,ignore_index=True)
fit_x,valid_x=best_features(raw_fit,raw_valid);layer1=TabularPredictor.load(str(source_run/'layer1_full'))
valid_prediction,valid_disagreement=predictions_and_disagreement(layer1,valid_x)
valid_meta=valid_x.copy();valid_meta['layer1_prediction']=valid_prediction;valid_meta['layer1_disagreement']=valid_disagreement;valid_meta['segment']=valid_meta.FLAT_TYPE.map(residual_segment)
valid_y=raw_valid[TARGET].astype(float).reset_index(drop=True)
display(oof.groupby('segment').agg(rows=(RESIDUAL_TARGET,'size'),mean_residual=(RESIDUAL_TARGET,'mean'),residual_RMSE=(RESIDUAL_TARGET,lambda values:np.sqrt(np.mean(values**2)))))


[categorical_clean] HDB lookup | TOWN: 26 -> 26 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | BLOCK: 2735 -> 2735 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | STREET: 576 -> 576 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | POSTAL_CODE: 9180 -> 9180 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | SUBZONE: 156 -> 156 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | PLANNING_AREA: 31 -> 31 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces


,rows,mean_residual,residual_RMSE
segment,,,
1_to_3_room,30131,171.397193,544.818609
4_room,31148,221.210220,628.698645
5_room,20469,261.662051,653.382544
executive,4675,288.241122,720.336394


In [3]:
meta_features=BEST_FEATURES+['layer1_prediction','layer1_disagreement']
train_oof=oof.loc[oof.fold<oof.fold.max()].reset_index(drop=True);tune_oof=oof.loc[oof.fold.eq(oof.fold.max())].reset_index(drop=True)
def fit_residual_expert(frame,segment,path):
    subset=frame.loc[frame.segment.eq(segment)]
    return TabularPredictor(label=RESIDUAL_TARGET,problem_type='regression',eval_metric='root_mean_squared_error',path=str(path)).fit(train_data=subset[meta_features+[RESIDUAL_TARGET]],hyperparameters=HYPERPARAMETERS,presets=['medium_quality','ignore_text'],time_limit=TIME_LIMIT,verbosity=2)
segments=['executive','5_room','4_room','1_to_3_room'];tuning_models={};segment_weights={};weight_rows=[]
for segment in segments:
    model=fit_residual_expert(train_oof,segment,output_root/f'tuning_{segment}');tuning_models[segment]=model
    tune_subset=tune_oof.loc[tune_oof.segment.eq(segment)];residual_prediction=model.predict(tune_subset[meta_features]).astype(float).to_numpy();actual_residual=tune_subset[RESIDUAL_TARGET].to_numpy()
    denominator=np.dot(residual_prediction,residual_prediction);analytic=0.0 if denominator==0 else np.dot(actual_residual,residual_prediction)/denominator
    candidates=np.unique(np.concatenate([np.linspace(0,1.5,151),[np.clip(analytic,0,1.5)]]));scores=[mean_squared_error(actual_residual,weight*residual_prediction)**.5 for weight in candidates]
    weight=float(candidates[np.argmin(scores)]);segment_weights[segment]=weight
    weight_rows.append({'segment':segment,'train_rows':int(train_oof.segment.eq(segment).sum()),'tune_rows':len(tune_subset),'weight':weight,'uncorrected_residual_RMSE':mean_squared_error(actual_residual,np.zeros_like(actual_residual))**.5,'corrected_residual_RMSE':min(scores)})
display(pd.DataFrame(weight_rows))


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       4.86 GB / 16.00 GB (30.4%)
Disk Space Avail:   166.91 GB / 460.38 GB (36.3%)
Presets specified: ['medium_quality', 'ignore_text']
Beginning AutoGluon training ... Time limit = 300s
AutoGluon will save models to "/Users/metul/Desktop/KDDM-project/exploration/metul/ag_exp_15_flat_type_residuals/20261016_182615/tuning_executive"
Train Data Rows:    3104
Train Data Columns: 20
Label Column:       layer1_residual
Problem Type:       regression
Preprocessing data...
Using Feature 

,segment,train_rows,tune_rows,weight,uncorrected_residual_RMSE,corrected_residual_RMSE
0,executive,3104,1571,0.199229,663.948554,653.967174
1,5_room,13463,7006,0.114456,584.588826,580.007565
2,4_room,20219,10929,0.110620,571.904488,569.590199
3,1_to_3_room,19586,10545,0.086726,521.661177,520.685990


In [4]:
final_models={};unweighted_correction=np.zeros(len(valid_meta));weighted_correction=np.zeros(len(valid_meta))
for segment in segments:
    model=fit_residual_expert(oof,segment,output_root/f'final_{segment}');final_models[segment]=model
    mask=valid_meta.segment.eq(segment).to_numpy();prediction=model.predict(valid_meta.loc[mask,meta_features]).astype(float).to_numpy()
    unweighted_correction[mask]=prediction;weighted_correction[mask]=segment_weights[segment]*prediction
unweighted_prediction=np.clip(valid_prediction+unweighted_correction,a_min=0,a_max=None);weighted_prediction=np.clip(valid_prediction+weighted_correction,a_min=0,a_max=None)
def metric(name,prediction):return {'model':name,'RMSE':mean_squared_error(valid_y,prediction)**.5,'MAE':mean_absolute_error(valid_y,prediction),'R2':r2_score(valid_y,prediction)}
comparison=pd.DataFrame([metric('Layer 1 AutoGluon',valid_prediction),metric('Unweighted flat-type residual experts',unweighted_prediction),metric('Weighted flat-type residual experts',weighted_prediction)])
display(comparison.sort_values('RMSE'))
rows=[]
for segment in segments:
    mask=valid_meta.segment.eq(segment).to_numpy();rows.append({'segment':segment,'rows':int(mask.sum()),'Layer1_RMSE':mean_squared_error(valid_y[mask],valid_prediction[mask])**.5,'Weighted_RMSE':mean_squared_error(valid_y[mask],weighted_prediction[mask])**.5,'weight':segment_weights[segment]})
display(pd.DataFrame(rows).sort_values('Weighted_RMSE'))


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       5.11 GB / 16.00 GB (31.9%)
Disk Space Avail:   167.60 GB / 460.38 GB (36.4%)
Presets specified: ['medium_quality', 'ignore_text']
Beginning AutoGluon training ... Time limit = 300s
AutoGluon will save models to "/Users/metul/Desktop/KDDM-project/exploration/metul/ag_exp_15_flat_type_residuals/20261016_182615/final_executive"
Train Data Rows:    4675
Train Data Columns: 20
Label Column:       layer1_residual
Problem Type:       regression
Preprocessing data...
Using Feature G

,model,RMSE,MAE,R2
2,Weighted flat-type residual experts,468.067517,339.893324,0.492429
0,Layer 1 AutoGluon,468.271759,340.986810,0.491986
1,Unweighted flat-type residual experts,472.508584,337.031600,0.482752


,segment,rows,Layer1_RMSE,Weighted_RMSE,weight
3,1_to_3_room,11450,437.079647,436.856808,0.086726
2,4_room,11678,472.380743,472.007092,0.110620
1,5_room,7270,481.633730,481.994464,0.114456
0,executive,1614,578.341630,577.109151,0.199229


Each row is routed only by its cleaned `FLAT_TYPE`, which is known at inference. The saved segment weights must travel with the four expert predictors for production use. Unknown categories fall back to the `1_to_3_room` segment in this first experiment.